# Qwen 3.5 4B через llama.cpp

> Ноутбук предполагает, что он лежит в корне проекта рядом с папками `llama.cpp` и `models`.

## Запуск модели

In [1]:
from zemi import exp

server = exp.restart()

Работающий llama-server не найден
llama-server запущен и готов, PID: 2660


## Диалог с моделью

In [11]:
from pathlib import Path
from openai import OpenAI
from openpyxl import load_workbook


client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
    timeout=300.0,
)

# Если ноутбук и файлы лежат в одной папке, достаточно относительных путей.
# Для общей папки VMware можно указать, например:
# DATA_DIR = Path(r"Z:\VMShare")
DATA_DIR = Path.cwd() / "data/case01"

EXCEL_FILES = [
    DATA_DIR / "Отчет 1.xlsx",
    DATA_DIR / "Отчет 2.xlsx",
    DATA_DIR / "Отчет 3.xlsx",
]


def cell_to_text(value) -> str:
    """Преобразует значение Excel-ячейки в компактный текст."""
    if value is None:
        return ""

    # openpyxl возвращает даты как datetime/date, если в Excel задан формат даты.
    if hasattr(value, "strftime"):
        return value.strftime("%Y-%m-%d")

    return str(value).replace("\t", " ").replace("\r", " ").replace("\n", " ")


def excel_to_text(path: Path) -> str:
    """Читает все листы книги и возвращает их содержимое в TSV-виде."""
    if not path.is_file():
        raise FileNotFoundError(f"Excel-файл не найден: {path}")

    workbook = load_workbook(path, read_only=True, data_only=True)
    parts = [f"===== ФАЙЛ: {path.name} ====="]

    try:
        for worksheet in workbook.worksheets:
            parts.append(f"--- ЛИСТ: {worksheet.title} ---")

            for row in worksheet.iter_rows(values_only=True):
                values = [cell_to_text(value) for value in row]

                # Удаляем только пустые ячейки справа, сохраняя структуру строки.
                while values and values[-1] == "":
                    values.pop()

                if values:
                    parts.append("\t".join(values))
    finally:
        workbook.close()

    return "\n".join(parts)


excel_context = "\n\n".join(excel_to_text(path) for path in EXCEL_FILES)

print("Загружены файлы:")
for path in EXCEL_FILES:
    print(f"- {path.name}: {path.stat().st_size} байт")

print(f"\nРазмер подготовленного текста: {len(excel_context)} символов")

TASK = """
Обработай все переданные Excel-отчёты.

Для каждого файла извлеки:
- название города/филиала;
- дату выгрузки;
- руководителя филиала;
- строки таблицы с полями date, article, cost.

Не включай строку «Итого».
Верни только корректный JSON без Markdown и пояснений.
Формат результата:
{
  "reports": [
    {
      "source_file": "имя файла",
      "city": "город",
      "export_date": "YYYY-MM-DD",
      "manager": "ФИО",
      "transactions": [
        {"date": "YYYY-MM-DD", "article": "...", "cost": 0}
      ]
    }
  ]
}
""".strip()

print("\nОтправляю три Excel-файла модели...")

response = client.chat.completions.create(
    model="qwen3.5-4b",
    messages=[
        {
            "role": "system",
            "content": (
                "Ты аккуратно преобразуешь содержимое Excel-отчётов "
                "в строго структурированный JSON. Не выдумывай отсутствующие данные."
            ),
        },
        {
            "role": "user",
            "content": f"{TASK}\n\nДАННЫЕ EXCEL:\n{excel_context}",
        },
    ],
    temperature=0.0,
    max_tokens=2048,
)

text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print("\nОтвет модели:\n")
print(text)


Загружены файлы:
- Отчет 1.xlsx: 10365 байт
- Отчет 2.xlsx: 10291 байт
- Отчет 3.xlsx: 10530 байт

Размер подготовленного текста: 962 символов

Отправляю три Excel-файла модели...

Ответ модели:

{
  "reports": [
    {
      "source_file": "Отчет 1.xlsx",
      "city": "Москва",
      "export_date": "2026-04-30",
      "manager": "Иванов И.И.",
      "transactions": [
        {"date": "2026-03-14", "article": "15", "cost": 83204},
        {"date": "2026-03-24", "article": "61", "cost": 53307},
        {"date": "2026-03-09", "article": "56", "cost": 67750},
        {"date": "2026-05-17", "article": "65", "cost": 21224},
        {"date": "2026-04-21", "article": "78", "cost": 65206},
        {"date": "2026-03-27", "article": "61", "cost": 40616},
        {"date": "2026-02-14", "article": "14", "cost": 33626}
      ]
    },
    {
      "source_file": "Отчет 2.xlsx",
      "city": "Санкт-Петербург",
      "export_date": "2026-04-30",
      "manager": "Петров П.П.",
      "transactions": [


In [ ]:
exp.print_response(response)

{
  "reports": [
    {
      "source_file": "Отчет 1.xlsx",
      "city": "Москва",
      "export_date": "2026-04-30",
      "manager": "Иванов И.И.",
      "transactions": [
        {"date": "2026-03-14", "article": "15", "cost": 83204},
        {"date": "2026-03-24", "article": "61", "cost": 53307},
        {"date": "2026-03-09", "article": "56", "cost": 67750},
        {"date": "2026-05-17", "article": "65", "cost": 21224},
        {"date": "2026-04-21", "article": "78", "cost": 65206},
        {"date": "2026-03-27", "article": "61", "cost": 40616},
        {"date": "2026-02-14", "article": "14", "cost": 33626}
      ]
    },
    {
      "source_file": "Отчет 2.xlsx",
      "city": "Санкт-Петербург",
      "export_date": "2026-04-30",
      "manager": "Петров П.П.",
      "transactions": [
        {"date": "2026-01-27", "article": "21", "cost": 3515},
        {"date": "2026-05-11", "article": "48", "cost": 1969},
        {"date": "2026-02-18", "article": "33", "cost": 2964},
       

## Остановка модели

Выполни эту ячейку, когда модель больше не нужна.

In [ ]:
exp.stop()

Сервер успешно остановлен.
